In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split, GridSearchCV, ShuffleSplit
import warnings
warnings.filterwarnings('ignore')

In [ ]:
def compute_scatter_matrices(X, y):
    '''Computes the scatter matrices for a given dataset.'''

    n_features = X.shape[1]
    mu = np.mean(X, axis=0)

    W = np.zeros((n_features, n_features))
    B = np.zeros((n_features, n_features))

    # For regression, use residual variance instead of class scatter
    y_mean = np.mean(y)
    for i in range(len(y)):
        xi = X[i]
        diff_x = (xi - mu).reshape(-1, 1)
        W += diff_x @ diff_x.T
        B += (y[i] - y_mean) ** 2 * diff_x @ diff_x.T

    W += np.eye(n_features) * 1e-6  # Regularization to ensure W is invertible

    return W, B

In [ ]:
def compute_dann_transform(W, B, eps=1e-3):
    ''' Computes the DANN transformation matrix.'''

    eigvals, eigvecs = np.linalg.eigh(W)

    W_inv_sqrt = eigvecs @ np.diag(1/(np.sqrt(eigvals) + 1e-8)) @ eigvecs.T

    B_star = W_inv_sqrt @ B @ W_inv_sqrt

    sigma = W_inv_sqrt @ (B_star + eps*np.eye(W.shape[0])) @ W_inv_sqrt

    # Cholesky decomposition for L
    L = np.linalg.cholesky(sigma)

    return L

In [ ]:
import numpy as np
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.neighbors import NearestNeighbors


class LocalDANN(BaseEstimator, RegressorMixin):

    def __init__(self, n_neighbors=5, k0=50, eps=1e-3):
        self.n_neighbors = n_neighbors
        self.k0 = k0
        self.eps = eps

    def fit(self, X, y):
        self.X = np.asarray(X)
        self.y = np.asarray(y, dtype=float)

        # Precompute Euclidean neighbor structure
        self.nbrs = NearestNeighbors(n_neighbors=self.k0, metric='euclidean')
        self.nbrs.fit(self.X)

        return self

    def _compute_scatter(self, Xn, yn):
        d = Xn.shape[1]
        mu = np.mean(Xn, axis=0)
        y_mean = np.mean(yn)

        W = np.zeros((d, d))
        B = np.zeros((d, d))

        for i in range(len(yn)):
            xi = Xn[i]
            diff_x = (xi - mu).reshape(-1, 1)
            W += diff_x @ diff_x.T
            B += (yn[i] - y_mean) ** 2 * diff_x @ diff_x.T

        # regularization
        W += 1e-6 * np.eye(d)

        return W, B

    def _compute_metric(self, W, B):

        eigvals, eigvecs = np.linalg.eigh(W)
        W_inv_sqrt = eigvecs @ np.diag(1/np.sqrt(eigvals + 1e-10)) @ eigvecs.T

        B_star = W_inv_sqrt @ B @ W_inv_sqrt

        Sigma = W_inv_sqrt @ (B_star + self.eps * np.eye(W.shape[0])) @ W_inv_sqrt

        return Sigma

    def predict(self, Xq):

        Xq = np.asarray(Xq)
        preds = []

        for x in Xq:

            # Step 1: get k0 neighbors (Euclidean)
            _, idx = self.nbrs.kneighbors([x])
            idx = idx[0]

            Xn = self.X[idx]
            yn = self.y[idx]

            # Step 2: local scatter
            W, B = self._compute_scatter(Xn, yn)

            # Step 3: local metric
            Sigma = self._compute_metric(W, B)

            # Step 4: compute distances (Mahalanobis)
            diff = self.X - x
            dists = np.sum((diff @ Sigma) * diff, axis=1)

            # Step 5: final KNN — mean of neighbor targets (regression)
            nn_idx = np.argsort(dists)[:self.n_neighbors]
            pred = np.mean(self.y[nn_idx])
            preds.append(pred)

        return np.array(preds)

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# Theoretical Risk Functions  (regression analogue of Cover & Hart 1967)
# ──────────────────────────────────────────────────────────────────────────

def compute_r_star_x(model, X_query, y_query):
    """
    Conditional risk r*(x) at each query point x for regression.

    r*(x) = (y_true - y_pred)^2  — squared residual per point.

    Parameters
    ----------
    model   : dict with 'dist_model' and 'prob_model' (KNeighborsRegressor GridSearchCV)
    X_query : query points  (n_samples × n_features)
    y_query : true target values  (n_samples,)

    Returns
    -------
    r_star : ndarray of shape (n_samples,)
    """
    prob_model = model['prob_model']
    y_pred = prob_model.predict(X_query)
    r_star = (np.asarray(y_query) - y_pred) ** 2
    return r_star


def compute_R_star(model, X_query, y_query):
    """
    Expected risk R* = E[r*(x)] — mean squared error over query points.

    Parameters
    ----------
    model   : dict with 'dist_model' and 'prob_model'
    X_query : held-out query points
    y_query : true targets

    Returns
    -------
    R_star : float
    """
    return float(compute_r_star_x(model, X_query, y_query).mean())


def compute_mu_hat(model, X):
    """
    Proxy for mu(x): the model's predicted value at each point (regression).

    Parameters
    ----------
    model : dict with 'prob_model' (KNeighborsRegressor GridSearchCV)
    X     : feature matrix

    Returns
    -------
    mu_hat : ndarray of shape (n_samples,)
    """
    prob_model = model['prob_model']
    return prob_model.predict(X)   # (n_samples,)


def compute_bias_vector(model, X_query, K_i):
    """
    Local bias for machine i (regression):
        Bi(x) = |mu(x) - (1/Ki) sum_j mu(xi,j)|

    Parameters
    ----------
    model   : dict with 'dist_model' and 'prob_model'
    X_query : query points  (n_samples x n_features)
    K_i     : number of neighbours used by this machine

    Returns
    -------
    B_i : ndarray of shape (n_samples,)
    """
    dist_model = model['dist_model']
    knn = dist_model.best_estimator_

    # mu(x) — predicted value at each query point
    mu_x = compute_mu_hat(model, X_query)   # (n_samples,)

    # Retrieve the K_i nearest training-set neighbours for each query point
    distances, indices = knn.kneighbors(X_query, n_neighbors=K_i)

    # mu at every training point
    X_train_arr = knn._fit_X
    X_train_df  = pd.DataFrame(X_train_arr, columns=X_query.columns)
    mu_train    = compute_mu_hat(model, X_train_df)   # (n_train,)

    # Average mu over the K_i neighbours
    mu_neighbour_means = mu_train[indices].mean(axis=1)  # (n_samples,)

    B_i = np.abs(mu_x - mu_neighbour_means)  # (n_samples,)
    return B_i


print('Theoretical risk functions defined.')

In [ ]:
def create_models(server_datasets):

    best_neighbors = []
    best_mse_scores = []
    model_instance_list = []
    dataset_sizes = []

    for df_train, df_test in server_datasets:
        params1 = {"n_neighbors": np.arange(3, 51, 2)}
        knn = KNeighborsRegressor()
        model1 = GridSearchCV(knn, params1, scoring='neg_mean_squared_error', cv=2, n_jobs=-1)

        X_s_train = df_train.drop(['y'], axis=1).values
        y_s_train = df_train['y'].values
        X_s_test  = df_test.drop(['y'], axis=1).values
        y_s_test  = df_test['y'].values

        model1.fit(X_s_train, y_s_train)
        y_pred = model1.predict(X_s_test)

        best_neighbors.append(model1.best_params_['n_neighbors'])
        best_mse_scores.append(mean_squared_error(y_s_test, y_pred))
        model_instance_list.append({
            'dist_model': model1,
            'prob_model': model1
        })

        dataset_sizes.append(df_train.shape[0])

    return (best_neighbors, best_mse_scores, model_instance_list, dataset_sizes)

In [ ]:
class ModelPerformanceAnalyzer:
    def __init__(self, models, model_names=None):
        self.models = models
        self.model_names = model_names or [f"Model_{i}" for i in range(len(models))]
        self.performance_df = None

    def analyze_models(self, X_unseen, y_unseen):
        results = []
        for i, (model, name) in enumerate(zip(self.models, self.model_names)):
            model_metrics = self._analyze_single_model(model, name, i, X_unseen, y_unseen)
            results.append(model_metrics)
        self.performance_df = pd.DataFrame(results)
        return self.performance_df

    def _analyze_single_model(self, model, name, index, X_unseen, y_unseen):
        y_pred = model['prob_model'].predict(X_unseen)

        metrics = {
            'model_name': name,
            'model_index': index,
            'mse': mean_squared_error(y_unseen, y_pred),
            'r2': r2_score(y_unseen, y_pred),
            'predictions': y_pred.tolist()
        }

        distance_metrics = self._calculate_distance_metrics(model['dist_model'], X_unseen)
        metrics.update(distance_metrics)
        return metrics

    def _calculate_distance_metrics(self, model, X_unseen):
        """Calculate various distance-based metrics."""
        try:
            nbrs = model.best_estimator_
            distances, indices = nbrs.kneighbors(X_unseen)
            distances_no_self = distances

            point_medians = np.median(distances_no_self, axis=1)
            point_means   = np.mean(distances_no_self, axis=1)
            point_mins    = np.min(distances_no_self, axis=1)
            point_maxs    = np.max(distances_no_self, axis=1)

            return {
                'median_distance_all_points': np.median(point_medians),
                'mean_distance_all_points':   np.mean(point_means),
                'min_distance_all_points':    np.min(point_mins),
                'max_distance_all_points':    np.max(point_maxs),
                'distance_std':               np.std(distances_no_self),
            }
        except Exception as e:
            warnings.warn(f"Distance calculation failed for model: {e}")
            return {
                'median_distance_all_points': np.nan,
                'mean_distance_all_points':   np.nan,
                'min_distance_all_points':    np.nan,
                'max_distance_all_points':    np.nan,
                'distance_std':               np.nan,
            }

    def get_predictions_matrix(self, model_index):
        """Get predictions for a specific model as numpy array."""
        if self.performance_df is not None:
            return np.array(self.performance_df.loc[model_index, 'predictions'])
        return None

    def get_all_predictions(self):
        """Get all prediction arrays as a 2D array (models x samples)."""
        if self.performance_df is not None:
            return np.array([np.array(p) for p in self.performance_df['predictions']])
        return None

In [ ]:
def run_distributed_knn_simulation(data_train, data_test, n_simulations=10, n_servers=100, test_size=0.2):
    """
    Distributed KNN Regressor simulation.
    Aggregation is performed via distance-based scoring weights (unchanged approaches 1-5)
    plus the optimal W* weight (approach 6).
    """
    import scipy.linalg as la

    simulation_results = []
    all_medians = []
    all_means   = []

    for sim in range(n_simulations):
        print(f"Running simulation {sim+1}/{n_simulations}")

        # 1. Split data
        X_train_full   = data_train.drop(['y'], axis=1)
        y_train_full   = data_train['y']
        X_test_heldout = data_test.drop(['y'], axis=1)
        y_test_heldout = data_test['y']

        # 2. Generate server datasets (use ShuffleSplit for regression — no stratification)
        def generate_server_datasets(train_pool, test_pool, n_servers=100,
                                     min_samples=1000, max_samples=5000):
            server_datasets = []
            X_train_pool = train_pool.drop(columns=['y'])
            y_train_pool = train_pool['y']
            X_test_pool  = test_pool.drop(columns=['y'])
            y_test_pool  = test_pool['y']

            for seed in range(n_servers):
                train_size = np.random.randint(min_samples, max_samples)
                ss_train   = ShuffleSplit(n_splits=1, train_size=train_size, random_state=seed)
                train_idx, _ = next(ss_train.split(X_train_pool))
                server_train = train_pool.iloc[train_idx]

                test_size_s = int(0.3 * train_size)
                ss_test = ShuffleSplit(n_splits=1, train_size=test_size_s, random_state=seed)
                test_idx, _ = next(ss_test.split(X_test_pool))
                server_test = test_pool.iloc[test_idx]

                server_datasets.append((server_train, server_test))

            return server_datasets

        train_pool_df = pd.concat([X_train_full, y_train_full], axis=1)
        test_pool_df  = pd.concat([X_test_heldout, y_test_heldout], axis=1)

        server_datasets = generate_server_datasets(train_pool_df, test_pool_df, n_servers=n_servers)

        # 3. Train models
        best_neighbors, best_mse_scores, model_instance_list, dataset_sizes = create_models(server_datasets)

        metrics_df = pd.DataFrame({
            'optimal_k_value': best_neighbors,
            'best_mse_scores': best_mse_scores,
            'model_instance':  model_instance_list,
            'dataset_size':    dataset_sizes
        })
        # Filter out degenerate models (infinite MSE)
        metrics_df = metrics_df[np.isfinite(metrics_df['best_mse_scores'])].reset_index(drop=True)

        # 4. Test on held-out data
        n_test_samples = min(500, len(X_test_heldout))
        test_indices   = np.random.choice(len(X_test_heldout), size=n_test_samples, replace=False)
        X_test_samples = X_test_heldout.iloc[test_indices]
        y_test_true    = y_test_heldout.iloc[test_indices]

        # Use ModelPerformanceAnalyzer to get all metrics and predictions
        analyzer    = ModelPerformanceAnalyzer(metrics_df['model_instance'].tolist())
        analysis_df = analyzer.analyze_models(X_test_samples, y_test_true)

        # Merge results
        metrics_df = metrics_df.merge(
            analysis_df[['model_index', 'median_distance_all_points', 'mean_distance_all_points']],
            left_index=True,
            right_on='model_index'
        ).reset_index(drop=True)

        all_medians.extend(metrics_df['median_distance_all_points'].tolist())
        all_means.extend(metrics_df['mean_distance_all_points'].tolist())

        # 5. Define scoring functions (unchanged — same aggregation method)
        def arctan_score(distance, df):
            min_d = df['median_distance_all_points'].min()
            max_d = df['median_distance_all_points'].max()
            return 0.5 + (np.arctan(((max_d + min_d)/2) - distance) / np.pi)

        def tanh_score(distance, df):
            min_d = df['median_distance_all_points'].min()
            max_d = df['median_distance_all_points'].max()
            return 0.5 + 0.5 * (np.tanh(((max_d + min_d)/2) - distance))

        def sigmoid_score(distance, df):
            min_d = df['median_distance_all_points'].min()
            max_d = df['median_distance_all_points'].max()
            return 1 / (1 + 2 * np.exp(((max_d + min_d)/2) - distance))

        def relu_score(distance, df):
            mean      = df['mean_distance_all_points'].mean()
            deviation = df['mean_distance_all_points'].std()
            score     = mean + 2 * deviation - distance
            return score if score > 0 else 0

        metrics_df['arctan_score']  = metrics_df['median_distance_all_points'].apply(lambda x: arctan_score(x, metrics_df))
        metrics_df['tanh_score']    = metrics_df['median_distance_all_points'].apply(lambda x: tanh_score(x, metrics_df))
        metrics_df['sigmoid_score'] = metrics_df['median_distance_all_points'].apply(lambda x: sigmoid_score(x, metrics_df))
        metrics_df['relu_score']    = metrics_df['median_distance_all_points'].apply(lambda x: relu_score(x, metrics_df))

        # 6. Extract predictions from each model
        all_preds = analyzer.get_all_predictions()  # (M, n_query)

        # 7. Theoretical risk quantities for regression
        sim_models = metrics_df['model_instance'].tolist()
        sim_K_list = metrics_df['optimal_k_value'].tolist()
        M_sim      = len(sim_models)
        n_q        = len(X_test_samples)
        y_true_arr = np.asarray(y_test_true)

        def _r_star_x(model, X, y):
            """r*(x) = squared residual per point (regression)."""
            y_pred = model['prob_model'].predict(X)
            return (np.asarray(y) - y_pred) ** 2

        def _bias_scalar_gwls(model, X_query, K_i):
            """
            Scalar spatial bias bi using Gaussian-kernel Weighted Least Squares (GWLS).
            Adapted for regression: target is the raw continuous prediction mu(x).
            """
            knn        = model['dist_model'].best_estimator_
            prob_model = model['prob_model']
            X_train    = knn._fit_X
            X_q_arr    = X_query.values

            X_train_df  = pd.DataFrame(X_train, columns=X_query.columns)
            p_train     = prob_model.predict(X_train_df)   # (n_train,) — regression predictions

            distances, indices = knn.kneighbors(X_query, n_neighbors=K_i)

            b_scalars = []
            for q_idx in range(n_q):
                xq       = X_q_arr[q_idx]
                nbr_idx  = indices[q_idx]
                nbr_dist = distances[q_idx]

                X_nbr    = X_train[nbr_idx]
                p_nbr    = p_train[nbr_idx]   # (K_i,) regression targets

                X_mean   = X_nbr.mean(axis=0)
                p_mean   = p_nbr.mean()
                Xi_c     = X_nbr - X_mean
                yi_c     = p_nbr - p_mean

                # Gaussian kernel weights
                h        = np.median(nbr_dist) + 1e-8
                gk_w     = np.exp(-(nbr_dist ** 2) / (h ** 2))
                W_diag   = np.diag(gk_w)

                # WLS normal equations
                XtWX     = Xi_c.T @ W_diag @ Xi_c + 1e-6 * np.eye(Xi_c.shape[1])
                XtWy     = Xi_c.T @ (gk_w * yi_c)
                g_i      = np.linalg.solve(XtWX, XtWy)

                med_idx  = np.argmin(np.abs(nbr_dist - np.median(nbr_dist)))
                d_i      = X_nbr[med_idx] - xq

                b_scalars.append(float(g_i @ d_i))

            return float(np.mean(b_scalars))

        # r*(x) and R* per server
        r_star_x_list, R_star_list = [], []
        for model in sim_models:
            r_x = _r_star_x(model, X_test_samples, y_test_true)
            r_star_x_list.append(r_x)
            R_star_list.append(float(r_x.mean()))

        r_star_matrix = np.stack(r_star_x_list, axis=0)  # (M, n_query)
        R_star_vec    = np.array(R_star_list)             # (M,)

        # bi scalar bias per server
        b_vec = np.array([
            _bias_scalar_gwls(model, X_test_samples, K_i)
            for model, K_i in zip(sim_models, sim_K_list)
        ])  # (M,)

        # C = b bT — rank-1 outer product
        C = np.outer(b_vec, b_vec)  # (M, M)

        # V diagonal: Vii = R*i / Ki
        V = np.diag(R_star_vec / np.array(sim_K_list, dtype=float))

        # Optimal W* via Sherman-Morrison
        K_vec      = np.array(sim_K_list, dtype=float)
        V_inv_diag = K_vec / (R_star_vec + 1e-10)
        S11        = np.sum(V_inv_diag)
        Sb1        = float(b_vec @ V_inv_diag)
        Sbb        = float(b_vec @ (V_inv_diag * b_vec))

        denom_sm = S11 - (Sb1 ** 2) / (1.0 + Sbb)
        num_vec  = V_inv_diag * (1.0 - b_vec * Sb1 / (1.0 + Sbb))
        W_star   = num_vec / denom_sm
        W_star   = np.clip(W_star, 0.0, None)
        W_star  /= W_star.sum()

        R_global_mean = float(R_star_vec.mean())
        R_optimal     = R_global_mean + 1.0 / denom_sm

        B_matrix = b_vec[:, None] * np.ones((M_sim, n_q))

        # 8. Weighted predictions — approaches 1–5 (heuristic) + approach 6 (optimal W*)
        approaches = {
            'approach_1': 'arctan_score',
            'approach_2': 'dataset_size',
            'approach_3': 'tanh_score',
            'approach_4': 'sigmoid_score',
            'approach_5': 'relu_score'
        }

        approach_preds = {}
        for approach_name, weight_col in approaches.items():
            weights = metrics_df[weight_col].values.astype(float)
            w_sum   = weights.sum()
            if w_sum == 0:
                w_sum = 1.0
            # all_preds: (M, n_query) — weighted mean across models
            approach_preds[approach_name] = (all_preds * weights[:, None]).sum(axis=0) / w_sum

        # Approach 6 — optimal W* (already sums to 1)
        approach_preds['approach_6'] = (all_preds * W_star[:, None]).sum(axis=0)

        # 9. Calculate MSE and R2 for all approaches
        sim_result = {'simulation': sim}
        for approach_name, preds in approach_preds.items():
            sim_result[f'{approach_name}_mse'] = mean_squared_error(y_test_true, preds)
            sim_result[f'{approach_name}_r2']  = r2_score(y_test_true, preds)

        sim_result.update({
            'n_servers':           len(metrics_df),
            'avg_dataset_size':    metrics_df['dataset_size'].mean(),
            'avg_median_distance': metrics_df['median_distance_all_points'].mean(),
            # Theoretical quantities
            'r_star_per_server':   R_star_list,
            'r_star_x_matrix':     r_star_matrix.tolist(),
            'bias_matrix':         B_matrix.tolist(),
            'C_matrix':            C.tolist(),
            'V_diag':              np.diag(V).tolist(),
            'W_star':              W_star.tolist(),
            'mean_R_star':         R_global_mean,
            'min_R_star':          float(R_star_vec.min()),
            'max_R_star':          float(R_star_vec.max()),
            'optimal_R':           R_optimal,
        })

        simulation_results.append(sim_result)

    results_df = pd.DataFrame(simulation_results)
    return results_df, all_medians, all_means

In [ ]:
# Load your data
# data_train = pd.read_csv('/kaggle/input/datasets/abhirajraje/oelp-dataset/adult_preprocessed_train.csv')
# data_test = pd.read_csv('/kaggle/input/datasets/abhirajraje/oelp-dataset/adult_preprocessed_test.csv')

data_train = pd.read_csv('../data/traffic_preprocessed_train.csv')
data_test  = pd.read_csv('../data/traffic_preprocessed_test.csv')

# Run multiple simulations
results_df, all_medians, all_means = run_distributed_knn_simulation(
    data_train, data_test, n_simulations=5, n_servers=10
)

In [ ]:
print(results_df)

In [ ]:
# Find the best KNN Regressor on overall data
X_train = data_train.drop(['y'], axis=1)
y_train = data_train['y']
X_test  = data_test.drop(['y'], axis=1)
y_test  = data_test['y']

params1 = {"n_neighbors": np.arange(3, 31, 2)}
knn     = KNeighborsRegressor()
model   = GridSearchCV(knn, params1, scoring='neg_mean_squared_error', cv=2, n_jobs=-1)
model.fit(X_train, y_train)
y_pred  = model.predict(X_test)

buffer_model = {"dist_model": model, "prob_model": model}

R_star_global_model = compute_R_star(buffer_model, X_test, y_test)
k_global_model      = model.best_params_['n_neighbors']
V_global_model      = R_star_global_model / k_global_model

benchmark_mse = mean_squared_error(y_test, y_pred)
benchmark_r2  = r2_score(y_test, y_pred)
benchmark_mae = mean_absolute_error(y_test, y_pred)

print(f"Best Params: {model.best_params_}")
print(f"Benchmark MSE:  {benchmark_mse:.4f}")
print(f"Benchmark MAE:  {benchmark_mae:.4f}")
print(f"Benchmark R²:   {benchmark_r2:.4f}")

In [ ]:
print("Simulation Results Summary:")
print("=" * 50)
print(f"Number of simulations: {len(results_df)}")
print(f"Average MSE Scores:")
print(f"Approach 1 (Arctan):  {results_df['approach_1_mse'].mean():.4f} ± {results_df['approach_1_mse'].std():.4f}")
print(f"Approach 2 (Size):    {results_df['approach_2_mse'].mean():.4f} ± {results_df['approach_2_mse'].std():.4f}")
print(f"Approach 3 (Tanh):    {results_df['approach_3_mse'].mean():.4f} ± {results_df['approach_3_mse'].std():.4f}")
print(f"Approach 4 (Sigmoid): {results_df['approach_4_mse'].mean():.4f} ± {results_df['approach_4_mse'].std():.4f}")
print(f"Approach 5 (ReLU):    {results_df['approach_5_mse'].mean():.4f} ± {results_df['approach_5_mse'].std():.4f}")
print(f"Approach 6 (W*):      {results_df['approach_6_mse'].mean():.4f} ± {results_df['approach_6_mse'].std():.4f}")
print()
print(f"Average R² Scores:")
print(f"Approach 1 (Arctan):  {results_df['approach_1_r2'].mean():.4f} ± {results_df['approach_1_r2'].std():.4f}")
print(f"Approach 2 (Size):    {results_df['approach_2_r2'].mean():.4f} ± {results_df['approach_2_r2'].std():.4f}")
print(f"Approach 3 (Tanh):    {results_df['approach_3_r2'].mean():.4f} ± {results_df['approach_3_r2'].std():.4f}")
print(f"Approach 4 (Sigmoid): {results_df['approach_4_r2'].mean():.4f} ± {results_df['approach_4_r2'].std():.4f}")
print(f"Approach 5 (ReLU):    {results_df['approach_5_r2'].mean():.4f} ± {results_df['approach_5_r2'].std():.4f}")
print(f"Approach 6 (W*):      {results_df['approach_6_r2'].mean():.4f} ± {results_df['approach_6_r2'].std():.4f}")
print()
print(f"Benchmark (global KNN Regressor) — MSE: {benchmark_mse:.4f}, R²: {benchmark_r2:.4f}")

In [ ]:
# %%
# Visualization
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
results_df[['approach_1_r2', 'approach_2_r2', 'approach_3_r2', 'approach_4_r2', 'approach_5_r2', 'approach_6_r2']].boxplot()
plt.title('R2 Score Distribution Across Simulations')
plt.ylabel('R2 Score')
plt.axhline(y=benchmark_r2, color='r', linestyle='--', label='Benchmark R2 Score')
plt.xticks([1, 2, 3, 4, 5, 6], ['Arctan', 'Size', 'Tanh', 'Sigmoid', 'RELU', 'W*'])

plt.subplot(1, 2, 2)
plt.plot(results_df['approach_1_r2'], label='Arctan', marker='o')
plt.plot(results_df['approach_2_r2'], label='Size', marker='s')
plt.plot(results_df['approach_3_r2'], label='Tanh', marker='^')
plt.plot(results_df['approach_4_r2'], label='Sigmoid', marker='X')
plt.plot(results_df['approach_5_r2'], label='RELU', marker='*')
plt.plot(results_df['approach_6_r2'], label='W*', marker='p')
plt.axhline(y=benchmark_r2, color='r', linestyle='--', label='Benchmark R2 Score')
plt.xticks(np.arange(len(results_df)), np.arange(1, len(results_df)+1))
plt.xlabel('Simulation')
plt.ylabel('R2 Score')
plt.title('R2 Score Trend Across Simulations')
plt.legend()
plt.grid(True)

# plt.savefig('f1_score_trends_std_normal.png')

plt.tight_layout()
plt.show()
